# Exercise 1: Creating a Movie Database Extracted from an API

In [1]:
# Libraries for HTTP requests and data manipulation
import requests
import pandas as pd 

# MySQL database connection
import mysql.connector
from mysql.connector import Error

# Used to clean API values
import numpy as np

# Environment variables (passwords, sensitive config)
import os 
from dotenv import load_dotenv
load_dotenv()
password_sql = os.getenv("PASS_SQL")

## Phase 1: Movie Data Extraction

In [3]:
# Fetches movie data from an API and returns a pandas DataFrame
def api_requests(url_api_movies):

    try:
        # Send a GET request to the API URL 
        movie_data = requests.get(url_api_movies)
        if movie_data.status_code == 200:
            print ("API connected")
            # Convert the JSON response into a pandas DataFrame
            df_movies = pd.DataFrame(movie_data.json())
            return df_movies
        else:
            # Handle cases where the server responds with an error code
            print ("API failed")

    # Handle connection-related errors (e.g., DNS failure, refused connection)        
    except requests.exceptions.ConnectionError as CnxE:
        print (CnxE)

    # Handle requests that exceed the timeout limit
    except requests.exceptions.Timeout as TO:
        print (TO)
    
    # Handle any other ambiguous exception that occurs while handling a request
    except requests.exceptions.RequestException as e:
        print (e)        
   

In [4]:
# Fetch data from the URL and store the returned DataFrame in df_movies
df_movies = api_requests("https://beta.adalab.es/resources/apis/pelis/pelis.json")
df_movies

API connected


,id,titulo,año,duracion,genero,adultos,subtitulos
0,1,The Godfather,1972,175,Crimen,False,"[es, en]"
1,2,The Godfather Part II,1974,202,Crimen,False,"[es, en]"
2,3,Pulp Fiction,1994,154,Crimen,True,"[es, en]"
3,4,Forrest Gump,1994,142,Drama,False,"[es, en, fr]"
4,5,The Dark Knight,2008,152,Acción,False,"[es, en]"
...,...,...,...,...,...,...,...
95,96,La vita è bella,1997,116,Drama,False,"[es, en, it]"
96,97,Requiem for a Dream,2000,102,Drama,True,"[es, en]"
97,98,Memento,2000,113,Thriller,True,"[es, en]"
98,99,Eternal Sunshine of the Spotless Mind,2004,108,Drama,False,"[es, en]"


## Phase 2: Database Creation

In [5]:
# Establishes a connection to the local MySQL server
def connection_mysql():

    try:
        connection = mysql.connector.connect(
            host = "127.0.0.1", 
            user= "root",
            password = password_sql,  # Uses the local environment password variable, not added in GitHUB
            # Note: We are creating a new database
        )
        print("Connection successful")
        return connection
    
    # Handle any database-related errors that occur during connection
    except Error as e:
        print (f"An error has occurred: {e}")

In [6]:
# Call the function to initialize the database connection and store it
connection = connection_mysql()

Connection successful


In [ ]:
# Define database and table names as variables for easier configuration and reuse
db_name = "movies_final_exercise" 
table_name = "movies_info"

In [ ]:
# Creates a new database if it does not already exist
def create_database(db_name):
    cursor = None
    try:
        # Create a cursor object to execute SQL commands
        cursor = connection.cursor ()
        # Define the SQL query using an f-string
        query = f"CREATE DATABASE IF NOT EXISTS {db_name}"
        # Execute the database creation query
        cursor.execute(query)
        print ("Query successful")

    # Catch any SQL execution errors
    except Error as e:
        print (f"Error creating database: {e}")
        
    # Ensure the cursor is always closed, even if an error occurs
    finally:
        if cursor != None:
            cursor.close()    

In [ ]:
create_database(db_name)

Query successful


## Phase 3: Inserting Data into the Database

In [23]:
df_movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          100 non-null    int64 
 1   titulo      100 non-null    str   
 2   año         100 non-null    int64 
 3   duracion    100 non-null    int64 
 4   genero      100 non-null    str   
 5   adultos     100 non-null    bool  
 6   subtitulos  100 non-null    object
dtypes: bool(1), int64(3), object(1), str(2)
memory usage: 4.9+ KB


In [ ]:
# Creates a specific table inside the designated database if it doesn't exist
def create_table(db_name, table_name):
    cursor = None
    try:
        # Create a cursor object to execute SQL commands
        cursor = connection.cursor ()
        # Select the database passed as an argument
        cursor.execute (f"USE {db_name};")
        # Define the SQL schema for the specified table
        query = f''' CREATE TABLE IF NOT EXISTS {table_name} (
                    id INT PRIMARY KEY AUTO_INCREMENT,
                    title VARCHAR(100) NOT NULL,
                    year YEAR,
                    runtime INT,
                    genre VARCHAR(30),
                    rating BOOL,
                    subtitles VARCHAR (30)
                    );'''
        # Execute the table creation query
        cursor.execute(query)
        print ("Query creation successful")
    
    # Catch any SQL execution errors
    except Error as e:
        print (f"Error creating table: {e}")

   # Ensure the cursor is always closed, even if an error occurs
    finally:
        if cursor != None:
            cursor.close() 

In [ ]:
create_table(db_name, table_name)

Query insert succesful


In [33]:
def erase_table(db_name, table_name):
    cursor = None
    try:
        user_respond = input (f"Table '{table_name}' will be deleted. Are you sure? (Y/N):").upper()
        # Check if the user confirmed the operation
        if user_respond == "Y":
            # Create a cursor object to execute SQL commands
            cursor = connection.cursor ()
            # Select the database passed as an argument
            cursor.execute (f"USE {db_name};")
            # Define the SQL query to drop the table if it exists
            query = f'''DROP TABLE IF EXISTS {table_name}'''
            # Execute the drop table query
            cursor.execute(query)
            connection.commit()
            print ("Table deleted successfully")
        else: 
            print ("Operation cancelled")

    # Catch any SQL execution errors
    except Error as e:
        print (f"Error deleting table: {e}")

    # Ensure the cursor is always closed, even if an error occurs
    finally:
        if cursor != None:
            cursor.close() 

In [ ]:
erase_table(db_name, table_name)

Table deleted successfully


In [ ]:
create_table(db_name, table_name)

Query creation successful


In [ ]:
# Inserts multiple rows of clean data from a DataFrame into the specified table
def insert_data(db_name, table_name):
    cursor = None
    try:
        # Create a cursor object to execute SQL commands
        cursor = connection.cursor ()
        # Select the database passed as an argument
        cursor.execute (f"USE {db_name};")
        # Define the parameterized SQL query for batch insertion
        query = f'''INSERT INTO {table_name} (title, year, runtime, 
                genre, rating, subtitles)
                VALUES (%s, %s, %s, %s, %s, %s) '''
        # Clean the DataFrame by replacing all variations of NaN with None (SQL NULL). 
        # String data and reorder columns
        df_clean = df_movies.replace({np.nan: None, 'nan': None, 'Nan': None})
        df_clean["subtitulos"] = df_clean["subtitulos"].astype(str)
        df_clean = df_clean[["titulo", "año", "duracion", "genero", "adultos", "subtitulos"]]

        # Convert the DataFrame rows into a list of tuples for the database
        values = df_clean.values.tolist() 

        # Execute the batch insertion of all rows at once
        cursor.executemany(query, values)
        connection.commit()  
        print(f"Successfully inserted {cursor.rowcount} new records")
    
     # Catch any SQL execution errors
    except Error as e:
        print (f"Error inserting data: {e}")
        
    # Ensure the cursor is always closed, even if an error occurs
    finally:
        if cursor != None:
            cursor.close() 
        

In [ ]:
insert_data(db_name, table_name)

Se han insertado 100 valores nuevos


## Phase 4: Querying the Data

### 1. How many movies have a runtime longer than 120 minutes?

In [ ]:

def

### 2. How many movies include Spanish subtitles?

### 3. How many movies have adult content?

### 4. What is the oldest movie registered in the database?

### 5. Show the average movie runtime grouped by genre.

### 6. How many movies have been registered per year? Sort from highest to lowest.

### 7. Which year has the highest number of movies in the database?

### 8. Get a list of all genres and the number of movies corresponding to each one.

### 9. Show all movies whose title contains the word "Godfather".